# Introduction to Deep Learning with TensorFlow & Keras

Build, train, and evaluate neural networks for regression and classification.


In [ ]:
# Colab setup
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Running in Google Colab - TensorFlow pre-installed!')
    from google.colab import drive
    drive.mount('/content/drive')
    for REPO in ['/content/drive/MyDrive/data-science-mastery', '/content/data-science-mastery']:
        if os.path.isdir(REPO):
            os.chdir(REPO)
            break
    print(f'Working dir: {os.getcwd()}')


## 1. Regression: House Price Prediction


In [ ]:
df = pd.read_csv('Data/house_pricing.csv')
X = pd.get_dummies(df.drop('price', axis=1), columns=['location'], drop_first=True)
y = df['price']

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print(f'Training samples: {X_train.shape[0]}, Features: {X_train.shape[1]}')


### Build a Neural Network


In [ ]:
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)  # regression output
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(
    X_train_sc, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training History'); axes[0].legend()
axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE')
axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
y_pred = model.predict(X_test_sc, verbose=0).flatten()
print(f'R2 Score:  {r2_score(y_test, y_pred):.4f}')
print(f'MAE:      ${mean_absolute_error(y_test, y_pred):,.2f}')


## 2. Classification: Customer Churn


In [ ]:
df_churn = pd.read_csv('Data/customer_churn.csv')
cat_cols = ['contract_type', 'internet_service', 'payment_method']
num_cols = ['tenure_months', 'monthly_charges', 'total_charges']
X_clf = pd.get_dummies(df_churn[cat_cols + num_cols], columns=cat_cols, drop_first=True)
y_clf = df_churn['churn']

X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)


In [ ]:
clf_model = models.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_tr.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

clf_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC()])

clf_history = clf_model.fit(
    X_tr_sc, y_tr,
    validation_split=0.2, epochs=100, batch_size=32,
    callbacks=[callbacks.EarlyStopping(patience=10)], verbose=0
)

from sklearn.metrics import classification_report
y_prob = clf_model.predict(X_te_sc, verbose=0).flatten()
y_pred_clf = (y_prob > 0.5).astype(int)
print(classification_report(y_te, y_pred_clf))


## 3. Save & Load Model


In [ ]:
model.save('house_price_model.keras')
print('Model saved as house_price_model.keras')

loaded = models.load_model('house_price_model.keras')
print(f'Loaded model verified: R2 = {r2_score(y_test, loaded.predict(X_test_sc, verbose=0).flatten()):.4f}')
